### ESG-BERT Classification

In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# 1️⃣ 定義 10 個模型名稱
model_names = [
    "ESGBERT/EnvironmentalBERT-forest",
    "ESGBERT/EnvironmentalBERT-action",
    "ESGBERT/EnvironmentalBERT-environmental",
    # "ESGBERT/SocRoBERTa-social",
    # "ESGBERT/GovRoBERTa-governance",
    "ESGBERT/GovernanceBERT-governance",
    # "ESGBERT/EnvRoBERTa-environmental",
    "ESGBERT/SocialBERT-social",
    "ESGBERT/EnvironmentalBERT-water",
    "ESGBERT/EnvironmentalBERT-biodiversity",
]

# 2️⃣ 預先載入所有 Tokenizer 和 Model
print("Loading all models... (This may take a while)")
tokenizers = {}
models = {}

for name in model_names:
    print(f"Loading {name}...")
    tokenizers[name] = AutoTokenizer.from_pretrained(name)
    models[name] = AutoModelForSequenceClassification.from_pretrained(name)

print("All models loaded successfully!\n")

Loading all models... (This may take a while)
Loading ESGBERT/EnvironmentalBERT-forest...
Loading ESGBERT/EnvironmentalBERT-action...
Loading ESGBERT/EnvironmentalBERT-environmental...
Loading ESGBERT/GovernanceBERT-governance...
Loading ESGBERT/SocialBERT-social...
Loading ESGBERT/EnvironmentalBERT-water...
Loading ESGBERT/EnvironmentalBERT-biodiversity...
All models loaded successfully!



In [ ]:
def split_long_text(text, tokenizer, max_length=512):
    """切割過長的文本，使每段的 tokens 數量不超過 max_length"""
    sentences = text.split(".")  # 先以句點切割
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        tokenized_sentence = tokenizer(sentence, add_special_tokens=False)
        token_count = len(tokenized_sentence["input_ids"])

        if token_count > max_length - 2:  # 單一個句子就超過 max_length
            words = sentence.split(",")  # 改用空格進一步切割
            temp_chunk = []
            temp_length = 0
            for word in words:
                word_tokens = tokenizer(word, add_special_tokens=False)["input_ids"]
                word_count = len(word_tokens)

                if temp_length + word_count > max_length - 2:
                    chunks.append(" ".join(temp_chunk))
                    temp_chunk = []
                    temp_length = 0

                temp_chunk.append(word)
                temp_length += word_count

            if temp_chunk:
                chunks.append(" ".join(temp_chunk))
        else:
            if current_length + token_count > max_length - 2:
                chunks.append(".".join(current_chunk))
                current_chunk = []
                current_length = 0

            current_chunk.append(sentence)
            current_length += token_count

    if current_chunk:
        chunks.append(".".join(current_chunk))

    return chunks

In [ ]:
def esg_classification(csr_data):
    results = {name: [] for name in model_names}

    for page_num, text in csr_data.items():
        print(f"Processing Page {page_num}...")

        for model_name in model_names:
            tokenizer = tokenizers[model_name]
            model = models[model_name]

            # 使用改進的文本切割函數
            text_chunks = split_long_text(text, tokenizer, max_length=512)

            page_results = []

            for chunk in text_chunks:
                inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)

                with torch.no_grad():
                    outputs = model(**inputs)

                logits = outputs.logits
                probs = torch.nn.functional.softmax(logits, dim=-1)  # 轉換為機率
                page_results.append(probs.numpy().tolist()[0])  # 轉成 list 存入

            results[model_name].extend(page_results)
    
    # 計算每個模型的分類結果平均值
    final_scores = {
        model_name: torch.tensor(results[model_name]).mean(dim=0).tolist()
        for model_name in model_names
    }

    return final_scores

Processing Page 1...
Processing Page 2...
Processing Page 3...
Processing Page 4...
Processing Page 5...
Processing Page 6...
Processing Page 7...
Processing Page 8...
Processing Page 9...
Processing Page 10...
Processing Page 11...
Processing Page 12...
Processing Page 13...
Processing Page 14...
Processing Page 15...
Processing Page 16...
Processing Page 17...
Processing Page 18...
Processing Page 20...
Processing Page 21...
Processing Page 22...
Processing Page 23...
Processing Page 24...
Processing Page 25...
Processing Page 26...
Processing Page 27...
Processing Page 28...
Processing Page 29...
Processing Page 30...
Processing Page 31...
Processing Page 32...
Processing Page 33...
Processing Page 34...
Processing Page 35...
Processing Page 36...
Processing Page 37...
Processing Page 38...
Processing Page 39...
Processing Page 40...
Processing Page 41...
Processing Page 42...
Processing Page 43...
Processing Page 44...
Processing Page 45...
Processing Page 46...
Processing Page 47.

### Perspective Taking

In [ ]:
import json
import nltk
from nltk.tokenize import word_tokenize

In [ ]:
# 確保 NLTK 需要的資源已下載
# nltk.download('punkt')
# nltk.data.path.append('/home/francia/anaconda3/envs/csr_env/nltk_data')

In [ ]:
# 定義第一人稱與第二人稱代詞
first_person_pronouns = {
    "I", "i", "I'm", "i'm", "Im", "im", "I'am", "i'am",
    "Me", "me", "My", "my", "Mine", "mine",
    "We", "we", "Us", "us", "Our", "our", "Ours", "ours",
    "I've", "i've", "I'd", "i'd", "I'll", "i'll",
    "We're", "we're", "We've", "we've", "We'd", "we'd", "We'll", "we'll",
    "I'd've", "i'd've", "We'll've", "we'll've", "We've'll", "we've'll"
}

second_person_pronouns = {
    "You", "you", "Your", "your", "Yours", "yours",
    "You're", "you're", "You've", "you've", "You'll", "you'll", "You'd", "you'd"
}

In [ ]:
# 計算 Perspective Taking 值
def calculate_perspective_taking(text):
    words = word_tokenize(text.lower())

    # 計算代詞數量
    first_person_count = sum(1 for word in words if word in first_person_pronouns)
    second_person_count = sum(1 for word in words if word in second_person_pronouns)

    # 公式計算
    perspective_taking = second_person_count / (first_person_count + second_person_count + 0.0001)
    
    return perspective_taking

Perspective Taking Score: 0.012787722149907654


[nltk_data] Downloading package punkt to /home/francia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Readability

In [ ]:
import textstat

def readability_metrics(text):
    fog = textstat.gunning_fog(text)
    flesch_reading_ease = textstat.flesch_reading_ease(text)
    smog = textstat.smog_index(text)
    automated_readability = textstat.automated_readability_index(text)
    
    return fog, flesch_reading_ease, smog, automated_readability

Gunning Fog Index: 12.68
Flesch Reading Ease: 21.43
SMOG Index: 17.4
Automated Readability Index: 17.8


### Sentiment analysis

In [ ]:
from textblob import TextBlob

In [ ]:
def sentiment_analysis(csr_text):
    blob = TextBlob(csr_text)
    polarity = blob.sentiment.polarity  # 介於 -1.0 到 1.0 之間
    subjectivity = blob.sentiment.subjectivity  # 介於 0.0 到 1.0 之間
    return polarity, subjectivity

Sentiment Polarity: 0.14706745930883836
Sentiment Subjectivity: 0.4075103474241407


### Repoert Length

In [ ]:
import math
# get the number of words by log
def report_length(csr_text):
    length = math.log(len(csr_text))
    return length

Log of Number of Words: 11.477930211972966


In [ ]:
# 讀取 JSON 檔案
json_path = os.path.abspath("/home/francia/research_hub/csr_project/CSR_report_processed_v4/NASDAQ/NASDAQ_REGN_2017/NASDAQ_REGN_2017_v1_preprocessed_tool.json")
with open(json_path, "r", encoding="utf-8") as f:
    csr_data = json.load(f)
csr_text = "".join(csr_data.values())